# JMA GSM GRIB2 Downloader

**Developed By Hossein Shokoohi & Hossein Mastaneh at Bushehr Meteorological Office. Bushehrmet.ir**

Professional downloader for WIS-JMA GSM files.
Set credentials and options in the **CONFIG** cell, then run all cells.

* Output folder is named by the **actual run date** (not today).
* Re-runs are resumable: existing, valid files are skipped.
* Downloads go to a `.part` temp file, size-checked, then moved into place
  (so interrupted downloads / HTML error pages never leave a corrupt .bin).
* HTTP 404 = 'not published yet' (not retried); network errors are retried.

In [ ]:
import os, time
from datetime import datetime, timedelta, timezone
import requests

# ====================== CONFIG ======================
# Enter your own WIS-JMA account credentials below.
# Register at https://www.wis-jma.go.jp/ to obtain a username and password.
USERNAME = ''   # your WIS-JMA username
PASSWORD = ''   # your WIS-JMA password

# Default save location (used if you just press Enter at the prompt).
# Change this to any folder on your own machine.
OUTPUT_ROOT_DEFAULT = r'C:\JMA-Data'

# Ask where to save. Press Enter to accept the default above.
_entered = input(f'Save location [{OUTPUT_ROOT_DEFAULT}]: ').strip().strip('\"').strip("'")
OUTPUT_ROOT = _entered if _entered else OUTPUT_ROOT_DEFAULT
print('Saving under:', OUTPUT_ROOT)

# 'all' for surface + every pressure level, or a list e.g. ['surface','850','500']
LEVELS = 'all'

# Run selection: set DATE/RUN to force a specific cycle, or leave both None for auto.
DATE = None        # e.g. '20260607'  (YYYYMMDD)
RUN  = None        # e.g. '06'        (one of '00','06','12','18')

MIN_VALID_BYTES = 1024     # discard files smaller than this (error pages etc.)
MAX_RETRIES = 4
RETRY_DELAY = 20           # seconds between retries
TIMEOUT = 120              # seconds per request
# ====================================================

# ---- dataset description (URLs + forecast steps) ----
BASE = 'https://www.wis-jma.go.jp/d/c/RJTD/GRIB/Global_Spectral_Model/Latitude_Longitude'
DOMAIN = '90.0_-5.0_30.0_195.0'

LEVEL_URLS = {
    'surface': f'{BASE}/0.25_0.25/{DOMAIN}/Surface_layers',
    '925':     f'{BASE}/0.5_0.5/{DOMAIN}/925hPa',
    '850':     f'{BASE}/0.5_0.5/{DOMAIN}/850hPa',
    '700':     f'{BASE}/0.5_0.5/{DOMAIN}/700hPa',
    '600':     f'{BASE}/0.5_0.5/{DOMAIN}/600hPa',
    '500':     f'{BASE}/0.5_0.5/{DOMAIN}/500hPa',
    '300':     f'{BASE}/0.5_0.5/{DOMAIN}/300hPa',
    '200':     f'{BASE}/0.5_0.5/{DOMAIN}/200hPa',
}

STEPS_00_12 = [
    '0000','0003','0006','0009','0012','0015','0018','0021',
    '0100','0103','0106','0109','0112','0115','0118','0121',
    '0200','0203','0206','0209','0212','0215','0218','0221',
    '0300','0303','0306','0309','0312','0315','0318','0321',
    '0400','0403','0406','0409','0412','0415','0418','0421',
    '0500','0503','0506','0509','0512','0518',
    '0600','0606','0612','0618',
    '0700','0706','0712','0718',
    '0800','0806','0812','0818',
    '0900','0906','0912','0918',
    '1000','1006','1012','1018',
    '1100',
]
STEPS_06_18 = [
    '0000','0003','0006','0009','0012','0015','0018','0021',
    '0100','0103','0106','0109','0112','0115','0118','0121',
    '0200','0203','0206','0209','0212','0215','0218','0221',
    '0300','0303','0306','0309','0312','0315','0318','0321',
    '0400','0403','0406','0409','0412','0415','0418','0421',
    '0500','0503','0506','0509','0512',
]

# ---- helpers ----
def auto_run_time():
    now = datetime.now(timezone.utc)
    h = now.hour
    if 7 <= h < 10:
        return now.replace(hour=0, minute=0, second=0, microsecond=0)
    elif 10 <= h < 19:
        return now.replace(hour=6, minute=0, second=0, microsecond=0)
    elif 19 <= h < 22:
        return now.replace(hour=12, minute=0, second=0, microsecond=0)
    elif 22 <= h < 24:
        return now.replace(hour=18, minute=0, second=0, microsecond=0)
    else:
        return (now - timedelta(days=1)).replace(hour=18, minute=0, second=0, microsecond=0)

def resolve_levels(levels):
    if isinstance(levels, str) and levels.strip().lower() == 'all':
        return list(LEVEL_URLS.keys())
    if isinstance(levels, str):
        levels = [x.strip() for x in levels.split(',')]
    out = []
    for tok in levels:
        if tok in LEVEL_URLS:
            out.append(tok)
        elif tok:
            print(f"  (ignoring unknown level '{tok}')")
    return out

def filenames_for(run_hh, level):
    steps = STEPS_00_12 if run_hh in ('00','12') else STEPS_06_18
    if level == 'surface':
        return [f'GSM_GPV_Rra2_Gll0p25deg_Lsurf_FD{v}_grib2.bin' for v in steps]
    return [f'GSM_GPV_Rra2_Gll0p5deg_Lp{level}_FD{v}_grib2.bin' for v in steps]

def download_one(url, dest, auth):
    if os.path.exists(dest) and os.path.getsize(dest) >= MIN_VALID_BYTES:
        return 'skip', os.path.getsize(dest)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(url, auth=auth, stream=True, timeout=TIMEOUT) as r:
                if r.status_code == 404:
                    return 'missing', 0
                r.raise_for_status()
                tmp = dest + '.part'
                size = 0
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk); size += len(chunk)
                if size < MIN_VALID_BYTES:
                    os.remove(tmp)
                    return 'failed', 0
                os.replace(tmp, dest)
                return 'ok', size
        except requests.exceptions.RequestException as e:
            print(f'      attempt {attempt}/{MAX_RETRIES} failed: {e}')
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    return 'failed', 0

# ---- run the download ----
import time as _time
auth = (USERNAME, PASSWORD)

if DATE and RUN:
    run_date, run_hh = DATE, RUN
else:
    t = auto_run_time()
    run_date = DATE or t.strftime('%Y%m%d')
    run_hh = RUN or t.strftime('%H')

levels = resolve_levels(LEVELS)
main_dir = f'{OUTPUT_ROOT}_{run_date}'
print(f'Run: {run_date} {run_hh}z   ->   {main_dir}\\{run_hh}\\<level>')
print(f'Levels: {", ".join(levels)}')

grand = {'ok': 0, 'skip': 0, 'missing': 0, 'failed': 0}
grand_bytes = 0
per_level = {}          # level -> dict(counts, bytes, nfiles, path)
failed_files = []       # (level, filename)
missing_files = []      # (level, filename)
start = _time.time()

for level in levels:
    out_dir = os.path.join(main_dir, run_hh, level)
    os.makedirs(out_dir, exist_ok=True)
    files = filenames_for(run_hh, level)
    base_url = LEVEL_URLS[level]
    counts = {'ok': 0, 'skip': 0, 'missing': 0, 'failed': 0}
    nbytes = 0
    print(f'\n[{level}]  {len(files)} files')
    for fn in files:
        url = f'{base_url}/{run_date}/{run_hh}0000/{fn}'
        dest = os.path.join(out_dir, fn)
        status, size = download_one(url, dest, auth)
        counts[status] += 1
        nbytes += size
        if status == 'failed':
            failed_files.append((level, fn))
        elif status == 'missing':
            missing_files.append((level, fn))
        tag = {'ok':'downloaded','skip':'exists','missing':'NOT PUBLISHED','failed':'FAILED'}[status]
        print(f'  {tag:<13} {fn}')
    print(f"  -> ok={counts['ok']} skip={counts['skip']} "
          f"missing={counts['missing']} failed={counts['failed']}  "
          f'({nbytes/1024/1024:.1f} MB)')
    for k in grand:
        grand[k] += counts[k]
    grand_bytes += nbytes
    per_level[level] = {'counts': counts, 'bytes': nbytes,
                        'nfiles': len(files), 'path': out_dir}

elapsed = _time.time() - start

# ---- detailed summary ----
from datetime import datetime, timezone

total_files = sum(v['nfiles'] for v in per_level.values())
saved_on_disk = grand['ok'] + grand['skip']

print('=' * 64)
print('DOWNLOAD SUMMARY')
print('=' * 64)
print(f"  Generated (UTC)  : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Model run        : {run_date}  {run_hh}z")
print(f"  Save location    : {main_dir}\\{run_hh}\\")
print(f"  Levels requested : {len(levels)}  ({', '.join(levels)})")
print(f"  Elapsed time     : {elapsed:.1f} s")
print()
print(f"  Files expected   : {total_files}")
print(f"  Downloaded (new) : {grand['ok']}")
print(f"  Skipped (exist)  : {grand['skip']}")
print(f"  Now on disk      : {saved_on_disk} / {total_files}")
print(f"  Not published    : {grand['missing']}")
print(f"  Failed (network) : {grand['failed']}")
print(f"  Total size       : {grand_bytes/1024/1024:.1f} MB "
      f"({grand_bytes/1024/1024/1024:.2f} GB)")
if elapsed > 0:
    print(f"  Avg speed        : {grand_bytes/1024/1024/elapsed:.1f} MB/s")

print()
print('-' * 64)
print('  PER-LEVEL BREAKDOWN')
print('-' * 64)
print(f"  {'level':<8} {'ok':>4} {'skip':>5} {'miss':>5} {'fail':>5} {'size(MB)':>9}   folder")
for level in levels:
    v = per_level[level]
    c = v['counts']
    print(f"  {level:<8} {c['ok']:>4} {c['skip']:>5} {c['missing']:>5} "
          f"{c['failed']:>5} {v['bytes']/1024/1024:>9.1f}   {v['path']}")

# list any problem files explicitly
if missing_files:
    print()
    print(f"  Not published on server ({len(missing_files)}):")
    for lvl, fn in missing_files:
        print(f"    [{lvl}] {fn}")
if failed_files:
    print()
    print(f"  FAILED — network errors ({len(failed_files)}):")
    for lvl, fn in failed_files:
        print(f"    [{lvl}] {fn}")
    print("  Re-run the download cell to retry just these; existing files are skipped.")

print()
if grand['failed'] == 0:
    print('  STATUS: complete ✅  (no network failures)')
else:
    print('  STATUS: finished with failures ⚠️  — re-run to retry.')
print('=' * 64)

# ---- write pointer file so plot notebooks know the exact run + location ----
# Written to the PARENT of OUTPUT_ROOT as 'jma_latest_run.json'.
# Example: OUTPUT_ROOT = ...\\Desktop\\JMA-Data  ->  pointer at ...\\Desktop\\jma_latest_run.json
import json as _json

def _pointer_path(output_root):
    parent = os.path.dirname(output_root) or '.'
    return os.path.join(parent, 'jma_latest_run.json')

if (grand['ok'] + grand['skip']) > 0:
    ptr = {
        'data_root': OUTPUT_ROOT,
        'run_date': run_date,
        'run_hour': run_hh,
        'written_utc': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
    }
    pp = _pointer_path(OUTPUT_ROOT)
    try:
        with open(pp, 'w') as f:
            _json.dump(ptr, f, indent=2)
        print('Pointer written:', pp)
        print(f'  -> run {run_date} {run_hh}z   data_root={OUTPUT_ROOT}')
    except Exception as e:
        print('Could not write pointer file:', e)
else:
    print('No files for this run — pointer not updated.')

# JMA GSM — 3-hourly Rain over Iran / Middle East

The files store precipitation **accumulated from run start** (`paramId 0`, `stepType accum`).
Since steps are uniformly 3-hourly, the rain in each 3 h window is:

    rain_3h(t) = accum(t) − accum(t−3)

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically. Order of preference:
#   1) pointer file written by the downloader (~/jma_latest_run.json)  [best]
#   2) otherwise, scan DEFAULT_DATA_ROOT for the newest downloaded run [fallback]
# So it works even if you haven't run the new downloader yet.
LEVEL   = 'surface'      # this notebook's level
PRODUCT = 'Rain_3hr'     # this notebook's output subfolder name

# Fallback location used only when the pointer file is absent.
# Set this to the SAME base path you save data to (before the _YYYYMMDD part).
DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '3-hourly Rain'
PARAM_UNIT = 'mm'

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

# Iran / Middle East zoom box
lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    """Newest <data_root>_<date>/<hour>/<level> that contains files."""
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if level == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    # 1) pointer
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    # 2) scan default root
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was {LEVEL} downloaded for this run?')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_accum_rain(path):
    """Return (accum, lats, lons, valid, run) for accumulated precip, or None."""
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            if a.get('GRIB_paramId') == 0 and a.get('GRIB_stepType') == 'accum':
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
                return np.asarray(da.values), lats, lons, valid, run
    return None
##%%
# ---- 3-hourly precipitation colormap (mm) ----
rain_levels = [0.1, 0.5, 1, 2, 4, 6, 10, 15, 20, 30, 40, 60, 80]
rain_colors = ['#c8e6ff','#7fb8ff','#3a8cff','#1f5fe0','#21d07a','#0fa84e',
               '#ffe14d','#ffac1c','#ff5e1c','#e01010','#b3007a','#8a00b3']
rain_cmap = mcolors.ListedColormap(rain_colors)
rain_cmap.set_under('#ffffff00')
rain_cmap.set_over('#5a00a0')
rain_norm = mcolors.BoundaryNorm(rain_levels, rain_cmap.N)
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- plot 3-hourly rain by differencing consecutive accumulations ----
def fmt_utc(t):
    """numpy datetime64 -> 'YYYY-MM-DD HH:00 UTC' (no minutes/seconds)."""
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG (pointer or DEFAULT_DATA_ROOT) and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
    print(f'{len(files)} surface files found')

prev_accum = None
prev_hour = None

for hour, path in files:
    rec = read_accum_rain(path)
    if rec is None:
        # +0h analysis: accumulation is zero -> set baseline and skip plotting
        prev_accum = None
        prev_hour = hour
        print(f'+{hour:03d}h : analysis (no accum field) — baseline')
        continue
    accum, lats, lons, valid, run = rec

    if prev_accum is None:
        # first window: accumulation since run start IS the 3h amount
        rain3h = accum.copy()
        win_start = prev_hour if prev_hour is not None else 0
    else:
        rain3h = accum - prev_accum
        win_start = prev_hour
    rain3h = np.clip(rain3h, 0, None)   # remove tiny negatives from rounding

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    if lons.ndim == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, rain3h, levels=rain_levels, cmap=rain_cmap,
                     norm=rain_norm, extend='both', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=rain_levels)
    cbar.set_label(CREDIT)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT})\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'rain3h_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{win_start:03d}h→+{hour:03d}h : saved {out}  (max {np.nanmax(rain3h):.1f} {PARAM_UNIT})')

    prev_accum = accum
    prev_hour = hour
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'rain_3hr.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 24-hour Rainfall over Iran / Middle East

Files store precipitation accumulated from run start (`paramId 0`, `stepType accum`).

24-hour rain = accum(t) − accum(t−24h). 

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically. Order of preference:
#   1) pointer file written by the downloader (~/jma_latest_run.json)  [best]
#   2) otherwise, scan DEFAULT_DATA_ROOT for the newest downloaded run [fallback]
LEVEL   = 'surface'      # this notebook's level (finds the files)
PRODUCT = 'Rain_24hr'    # output subfolder name

# Fallback location used only when the pointer file is absent.
# Set this to the SAME base path you save data to (before the _YYYYMMDD part).
DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '24-hour Rainfall'
PARAM_UNIT = 'mm'

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

# Iran / Middle East zoom box
lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    """Newest <data_root>_<date>/<hour>/<level> that contains files."""
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if level == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was {LEVEL} downloaded for this run?')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_rain(path):
    """Return (data, lats, lons, valid, run) for accumulated precip, or None."""
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            if a.get('GRIB_paramId') == 0 and a.get('GRIB_stepType') == 'accum':
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                lon2, lat2 = np.meshgrid(lons, lats)
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
                return np.asarray(da.values), lat2, lon2, valid, run
    return None
##%%
# ---- 24-hour precipitation colormap (mm) ----
rain_levels = [1, 2, 5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200]
rain_colors = ['#c8e6ff','#7fb8ff','#3a8cff','#1f5fe0','#21d07a','#0fa84e',
               '#ffe14d','#ffac1c','#ff5e1c','#e01010','#b3007a','#8a00b3']
rain_cmap = mcolors.ListedColormap(rain_colors)
rain_cmap.set_under('#ffffff00')
rain_cmap.set_over('#5a00a0')
rain_norm = mcolors.BoundaryNorm(rain_levels, rain_cmap.N)
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m',
                                   category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   '
      f'Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    # other countries thin grey
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    # sea coastline at 1.5 px
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    # Caspian Sea coastline at 1.5 px
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.5, zorder=4,
                          joinstyle='round', capstyle='round')
    # Iran provinces 1.2 px
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.2, zorder=5,
                          joinstyle='round', capstyle='round')
    # Iran national border 1.8 px on top
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none',
                      edgecolor='black', linewidth=1.8, zorder=6,
                      joinstyle='round', capstyle='round')
##%%
# ---- plot 24-hour rainfall: accum(t) - accum(t-24h) ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
print(f'{len(files)} surface files found')

# build a lookup: forecast hour -> accumulated field (only where rain exists)
accum_by_hour = {}
meta_by_hour = {}
for hour, path in files:
    rec = read_rain(path)
    if rec is None:
        continue
    data, lats, lons, valid, run = rec
    accum_by_hour[hour] = data
    meta_by_hour[hour] = (lats, lons, valid, run)

for hour in sorted(accum_by_hour):
    prev = hour - 24
    if prev < 0:
        print(f'+{hour:03d}h : window starts before run — skipping (need +24h history)')
        continue
    if prev == 0:
        # accumulation at +0h is zero, so 0->24h total is just accum(24)
        rain24 = accum_by_hour[hour]
    elif prev in accum_by_hour:
        rain24 = accum_by_hour[hour] - accum_by_hour[prev]
    else:
        print(f'+{hour:03d}h : no matching +{prev:03d}h file — skipping')
        continue
    rain24 = np.clip(rain24, 0, None)
    lats, lons, valid, run = meta_by_hour[hour]

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, rain24, levels=rain_levels, cmap=rain_cmap,
                     norm=rain_norm, extend='both', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=rain_levels)
    cbar.set_label(CREDIT)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT})\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'rain24h_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{prev:03d}h→+{hour:03d}h : saved {out}  (max {np.nanmax(rain24):.1f} {PARAM_UNIT})')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'rain_24hr.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — Accumulated Rain over Iran / Middle East

Rain field = `paramId 0` + `stepType == 'accum'` (accumulated precipitation from run start, mm).

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL   = 'surface'           # finds the files
PRODUCT = 'Accumulated_Rain'  # output subfolder name

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = 'Accumulated Rain'
PARAM_UNIT = 'mm'

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if level == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was {LEVEL} downloaded for this run?')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_rain(path):
    """Return (data, lats, lons, valid, run) for accumulated precip, or None."""
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            if a.get('GRIB_paramId') == 0 and a.get('GRIB_stepType') == 'accum':
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                lon2, lat2 = np.meshgrid(lons, lats)
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
                return np.asarray(da.values), lat2, lon2, valid, run
    return None
##%%
# ---- precipitation colormap (mm) ----
rain_levels = [0.5, 1, 2, 5, 10, 15, 20, 30, 40, 60, 80, 100, 150, 200]
rain_colors = ['#c8e6ff','#7fb8ff','#3a8cff','#1f5fe0','#21d07a','#0fa84e',
               '#ffe14d','#ffac1c','#ff5e1c','#e01010','#b3007a','#8a00b3','#5a00a0']
rain_cmap = mcolors.ListedColormap(rain_colors)
rain_cmap.set_under('#ffffff00')
rain_cmap.set_over('#3a003a')
rain_norm = mcolors.BoundaryNorm(rain_levels, rain_cmap.N)
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m',
                                   category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   '
      f'Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    # other countries thin grey
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    # sea coastline at 1.5 px
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    # Caspian Sea coastline at 1.5 px
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.5, zorder=4,
                          joinstyle='round', capstyle='round')
    # Iran provinces 1.2 px
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.2, zorder=5,
                          joinstyle='round', capstyle='round')
    # Iran national border 1.8 px on top
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none',
                      edgecolor='black', linewidth=1.8, zorder=6,
                      joinstyle='round', capstyle='round')
##%%
# ---- plot accumulated rain for every forecast hour ----
def fmt_utc(t):
    """numpy datetime64 -> 'YYYY-MM-DD HH:00 UTC' (no minutes/seconds)."""
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
print(f'{len(files)} surface files found')

for hour, path in files:
    rec = read_rain(path)
    if rec is None:
        print(f'+{hour:03d}h : no accumulated-rain field (analysis), skipping')
        continue
    data, lats, lons, valid, run = rec

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, data, levels=rain_levels, cmap=rain_cmap,
                     norm=rain_norm, extend='both', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=rain_levels)
    cbar.set_label(CREDIT)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT})\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'rain_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (max {np.nanmax(data):.1f} {PARAM_UNIT})')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'accumulated_rain.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — Total Cloud Cover over Iran / Middle East

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL   = 'surface'
PRODUCT = 'Total_Cloud'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = 'Total Cloud Cover'
PARAM_UNIT = '%'

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if level == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was {LEVEL} downloaded for this run?')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_total_cloud(path):
    """Return (data, lats, lons, valid, run) for the total cloud cover field
    (paramId 0, stepType instant), or None if absent."""
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            if a.get('GRIB_paramId') == 0 and a.get('GRIB_stepType') == 'instant':
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                lon2, lat2 = np.meshgrid(lons, lats)
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
                return np.asarray(da.values), lat2, lon2, valid, run
    return None
##%%
# ---- total cloud colormap (%) ----
cloud_levels = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
cloud_colors = ['#2b6cd4','#4b86dd','#6fa0e4','#93b8ec','#b6cef2','#d4def5',
                '#dcdcdc','#c2c2c2','#a3a3a3','#838383']
cloud_cmap = mcolors.ListedColormap(cloud_colors)
cloud_norm = mcolors.BoundaryNorm(cloud_levels, cloud_cmap.N)
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- plot total cloud cover for every forecast hour ----
def fmt_utc(t):
    """numpy datetime64 -> 'YYYY-MM-DD HH:00 UTC' (no minutes/seconds)."""
    try:
        dt = np.datetime64(t, 'h')          # truncate to the hour
        s = str(dt).replace('T', ' ')
        return f'{s}:00 UTC'
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
print(f'{len(files)} surface files found')

for hour, path in files:
    rec = read_total_cloud(path)
    if rec is None:
        print(f'+{hour:03d}h : no total-cloud field, skipping')
        continue
    data, lats, lons, valid, run = rec

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, data, levels=cloud_levels, cmap=cloud_cmap,
                     norm=cloud_norm, extend='neither', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=cloud_levels)
    cbar.set_label(CREDIT)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT})\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'totalcloud_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (max {np.nanmax(data):.0f} %)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'total_cloud.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

## JMA GSM — 2 m Temperature (shaded) + 10 m Wind streamlines over Iran / Middle East

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL   = 'surface'
PRODUCT = 'Temp2m'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '2 m Temperature'
PARAM_UNIT = '°C'

STREAM_DENSITY = 2.5

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if level == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was {LEVEL} downloaded for this run?')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_temp_wind(path):
    """Return (t2m_C, u, v, lats, lons, valid, run); temperature in degC, wind m/s.
    Returns None if the 2 m temperature field is missing."""
    t2m = u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            pid = da.attrs.get('GRIB_paramId')
            if pid == 167:      # 2 m temperature (K)
                t2m = np.asarray(da.values) - 273.15
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif pid == 165:    # 10 m U
                u = np.asarray(da.values)
            elif pid == 166:    # 10 m V
                v = np.asarray(da.values)
    if t2m is None:
        return None
    return t2m, u, v, lats, lons, valid, run
##%%
# ---- temperature colormap (°C) ----
temp_levels = [-10, -5, 0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
temp_cmap = plt.get_cmap('RdYlBu_r')
temp_norm = mcolors.BoundaryNorm(temp_levels, temp_cmap.N)
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- plot 2 m temperature + wind streamlines for every forecast hour ----
def fmt_utc(t):
    """numpy datetime64 -> 'YYYY-MM-DD HH:00 UTC' (no minutes/seconds)."""
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
print(f'{len(files)} surface files found')

for hour, path in files:
    rec = read_temp_wind(path)
    if rec is None:
        print(f'+{hour:03d}h : 2 m temperature not found, skipping')
        continue
    t2m, u, v, lats, lons, valid, run = rec

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # shaded temperature (degC)
    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, t2m, levels=temp_levels, cmap=temp_cmap,
                     norm=temp_norm, extend='both', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=temp_levels)
    cbar.set_label(CREDIT)

    # 10 m wind streamlines (flow pattern). streamplot needs 1-D, strictly
    # increasing coords, so flip if latitudes run north->south.
    if u is not None and v is not None:
        lon1d, lat1d = lons, lats
        U, V = u, v
        if lat1d[0] > lat1d[-1]:
            lat1d = lat1d[::-1]
            U = U[::-1, :]
            V = V[::-1, :]
        ax.streamplot(lon1d, lat1d, U, V, density=STREAM_DENSITY,
                      linewidth=0.6, color='black', arrowsize=0.8,
                      transform=ccrs.PlateCarree())

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) + 10 m Wind streamlines\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'temp2m_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (min {np.nanmin(t2m):.1f} / max {np.nanmax(t2m):.1f} {PARAM_UNIT})')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'temp2m.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 10 m Wind + MSLP isobars + pressure centers (L/H)

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.ndimage import minimum_filter, maximum_filter
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL   = 'surface'
PRODUCT = 'Wind_MSLP'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '10 m Wind'
PARAM_UNIT = 'kt'
MS_TO_KT = 1.94384
BARB_STRIDE = 5            # plot a barb every Nth grid point
ISOBAR_STEP = 4           # hPa between isobar lines
LABEL_EVERY = 30          # label spacing along each isobar (lower = more labels)
CENTER_WINDOW = 25        # neighbourhood (grid points) for L/H detection

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if level == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was {LEVEL} downloaded for this run?')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    """Return dict with u_kt, v_kt, speed_kt, mslp_hpa, lats, lons, valid, run.
    Returns None if wind components are missing."""
    u = v = mslp = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            pid = da.attrs.get('GRIB_paramId')
            if pid == 165:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif pid == 166:
                v = np.asarray(da.values)
            elif pid == 260074:        # MSL pressure (Pa)
                mslp = np.asarray(da.values) / 100.0   # -> hPa
    if u is None or v is None:
        return None
    u_kt = u * MS_TO_KT
    v_kt = v * MS_TO_KT
    speed_kt = np.sqrt(u_kt**2 + v_kt**2)
    return dict(u_kt=u_kt, v_kt=v_kt, speed_kt=speed_kt, mslp=mslp,
                lats=lats, lons=lons, valid=valid, run=run)

def find_centers(field, lons, lats, size, kind,
                 box=None, min_sep_deg=3.0):
    """Local minima ('L') or maxima ('H') of `field`.
    - box=(lon_min,lon_max,lat_min,lat_max): restrict search to the map area
      so values/positions belong to what is shown.
    - min_sep_deg: merge centers closer than this (deg) and keep the most
      extreme one, so flat areas don't produce repeated stacked L/H.
    Returns list of (lon, lat, value)."""
    if field is None:
        return []
    if kind == 'L':
        ext = (minimum_filter(field, size=size) == field)
    else:
        ext = (maximum_filter(field, size=size) == field)

    cand = []
    yy, xx = np.where(ext)
    for j, i in zip(yy, xx):
        lon, lat, val = lons[i], lats[j], field[j, i]
        if box is not None:
            if not (box[0] <= lon <= box[1] and box[2] <= lat <= box[3]):
                continue
        cand.append((lon, lat, val))

    # sort so the most extreme go first (lowest for L, highest for H)
    cand.sort(key=lambda c: c[2], reverse=(kind == 'H'))

    kept = []
    for lon, lat, val in cand:
        if all((lon - kl)**2 + (lat - ka)**2 >= min_sep_deg**2
               for kl, ka, _ in kept):
            kept.append((lon, lat, val))
    return kept

##%%
# ---- wind speed colormap (knots) ----
wind_levels = [0, 5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70]
wind_colors = ['#e8f4ff','#c8e6ff','#9fd0ff','#7fb8ff','#5aa0f0','#3a8cff',
               '#21d07a','#ffe14d','#ffac1c','#ff5e1c','#e01010']
wind_cmap = mcolors.ListedColormap(wind_colors)
wind_cmap.set_over('#8a00b3')
wind_norm = mcolors.BoundaryNorm(wind_levels, wind_cmap.N)
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- plot wind + MSLP isobars + L/H centers for every forecast hour ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

def in_box(lon, lat):
    return (lon_min <= lon <= lon_max) and (lat_min <= lat <= lat_max)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
print(f'{len(files)} surface files found')

for hour, path in files:
    d = read_fields(path)
    if d is None:
        print(f'+{hour:03d}h : U/V wind not found, skipping')
        continue
    lons, lats = d['lons'], d['lats']

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # 1) shaded wind speed (kt)
    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, d['speed_kt'], levels=wind_levels, cmap=wind_cmap,
                     norm=wind_norm, extend='max', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=wind_levels)
    cbar.set_label(CREDIT)

    # 2) MSLP isobars
    if d['mslp'] is not None:
        # anchor levels to 1000 hPa so labels fall on standard values
        lo = np.floor((np.nanmin(d['mslp']) - 1000) / ISOBAR_STEP) * ISOBAR_STEP + 1000
        hi = np.ceil((np.nanmax(d['mslp']) - 1000) / ISOBAR_STEP) * ISOBAR_STEP + 1000
        clevs = np.arange(lo, hi + ISOBAR_STEP, ISOBAR_STEP)
        cs = ax.contour(lons, lats, d['mslp'], levels=clevs, colors='black',
                        linewidths=0.8, transform=ccrs.PlateCarree())
        # dense labels: place a number every LABEL_EVERY points along each
        # contour segment, so every isobar is labelled repeatedly (Pivotal style)
        label_pts = []
        for seg in cs.allsegs:
            for poly in seg:
                if len(poly) > LABEL_EVERY:
                    for k in range(LABEL_EVERY // 2, len(poly), LABEL_EVERY):
                        label_pts.append(tuple(poly[k]))
        if label_pts:
            ax.clabel(cs, fmt='%d', fontsize=6, inline=True, inline_spacing=2,
                      rightside_up=True, manual=label_pts)
        else:
            ax.clabel(cs, fmt='%d', fontsize=6, inline=True)

        # 3) L / H centers
        box = (lon_min, lon_max, lat_min, lat_max)
        lows = find_centers(d['mslp'], lons, lats, CENTER_WINDOW, 'L', box=box)
        highs = find_centers(d['mslp'], lons, lats, CENTER_WINDOW, 'H', box=box)
        for lon, lat, val in lows:
            if in_box(lon, lat):
                ax.text(lon, lat, 'L', color='red', fontsize=20, fontweight='bold',
                        ha='center', va='center', transform=ccrs.PlateCarree())
                ax.text(lon, lat - 0.6, f'{val:.0f}', color='red', fontsize=8,
                        ha='center', va='center', transform=ccrs.PlateCarree())
        for lon, lat, val in highs:
            if in_box(lon, lat):
                ax.text(lon, lat, 'H', color='blue', fontsize=20, fontweight='bold',
                        ha='center', va='center', transform=ccrs.PlateCarree())
                ax.text(lon, lat - 0.6, f'{val:.0f}', color='blue', fontsize=8,
                        ha='center', va='center', transform=ccrs.PlateCarree())

    # 4) wind barbs (kt), thinned
    s = BARB_STRIDE
    ax.barbs(lons[::s], lats[::s], d['u_kt'][::s, ::s], d['v_kt'][::s, ::s],
             length=5, linewidth=0.5, transform=ccrs.PlateCarree())

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}); isobars in hPa\n'
        f'Run Time: {fmt_utc(d["run"])}    Valid Time: {fmt_utc(d["valid"])}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'wind_mslp_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (max wind {np.nanmax(d["speed_kt"]):.1f} kt)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'wind_mslp.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 925 hPa Wind Streamlines over Iran / Middle East

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json

##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 925
PRODUCT   = 'Streamlines_925'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '925 hPa Wind'
PARAM_UNIT = 'kt'

LEVEL_LABEL = '925 hPa'
MS_TO_KT = 1.94384
STREAM_DENSITY = 2.5

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lp925_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_uv(path):
    """Return (u, v, lats, lons, valid, run) in m/s. Detects U/V by paramId
    (131/132 on pressure levels) or by CF/short name as a fallback."""
    u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            is_u = (pid == 131) or sn in ('u',) or cf == 'eastward_wind'
            is_v = (pid == 132) or sn in ('v',) or cf == 'northward_wind'
            if is_u and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif is_v and v is None:
                v = np.asarray(da.values)
    if u is None or v is None:
        return None
    return u, v, lats, lons, valid, run
##%%
# ---- (run once) inspect variables in a 925 file to confirm U/V ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_LABEL}')
if _files:
    _h, _p = next((h, p) for h, p in _files if h >= 0)
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} "
                  f"shortName={a.get('GRIB_shortName')} "
                  f"cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- wind speed colormap (knots) ----
wind_levels = [0, 5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70]
wind_colors = ['#e8f4ff','#c8e6ff','#9fd0ff','#7fb8ff','#5aa0f0','#3a8cff',
               '#21d07a','#ffe14d','#ffac1c','#ff5e1c','#e01010']
wind_cmap = mcolors.ListedColormap(wind_colors)
wind_cmap.set_over('#8a00b3')
wind_norm = mcolors.BoundaryNorm(wind_levels, wind_cmap.N)
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- plot 925 hPa streamlines for every forecast hour ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_uv(path)
    if rec is None:
        print(f'+{hour:03d}h : U/V not found, skipping')
        continue
    u, v, lats, lons, valid, run = rec
    u_kt = u * MS_TO_KT
    v_kt = v * MS_TO_KT
    speed_kt = np.sqrt(u_kt**2 + v_kt**2)

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # shaded wind speed (kt)
    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, speed_kt, levels=wind_levels, cmap=wind_cmap,
                     norm=wind_norm, extend='max', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=wind_levels)
    cbar.set_label(CREDIT)

    # streamlines (streamplot needs 1-D increasing coords; flip if N->S)
    lon1d, lat1d = lons, lats
    U, V = u_kt, v_kt
    if lat1d[0] > lat1d[-1]:
        lat1d = lat1d[::-1]
        U = U[::-1, :]
        V = V[::-1, :]
    ax.streamplot(lon1d, lat1d, U, V, density=STREAM_DENSITY,
                  linewidth=0.6, color='black', arrowsize=0.8,
                  transform=ccrs.PlateCarree())

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded) + Streamlines \n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'streamlines925_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (max {np.nanmax(speed_kt):.1f} {PARAM_UNIT})')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'streamlines_925.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 850 hPa Temperature Advection + Wind Streamlines

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 850
PRODUCT   = 'TempAdv_Streamlines_850'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '850 hPa Temperature Advection'
PARAM_UNIT = '°C/h'

MS_TO_KT = 1.94384
STREAM_DENSITY = 2.5
TEMP_CONTOUR_STEP = 2

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m',
                                   category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03     # deg, national border smoothing
PROV_SIMPLIFY   = 0.06     # deg, province smoothing (stronger -> no coastal clumps)
COAST_SIMPLIFY  = 0.03     # deg, coastline smoothing

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   '
      f'Caspian polys: {len(CASPIAN)}')
##%%
# ---- (diagnostic) only needed if provinces above show 0 ----
# Prints the attribute keys + a few sample values so we can see how Iran is named.
from collections import Counter
recs = list(shapereader.Reader(_provinces).records())
print('admin-1 attribute keys:', list(recs[0].attributes.keys()))
# show the distinct 'admin' values that contain 'Ira'
vals = Counter(str(r.attributes.get('admin','')) for r in recs)
print('\nadmin values containing "Ira":')
for k, c in vals.items():
    if 'Ira' in k:
        print(f'  {k!r}  ({c} polygons)')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    """Return (t_C, u, v, lats, lons, valid, run): temperature in degC, wind m/s.
    U/V by paramId 131/132 or CF name; temperature by paramId 130 ('t') or CF name."""
    t = u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 130 or sn == 't' or cf == 'air_temperature') and t is None:
                t = np.asarray(da.values)
    if u is None or v is None or t is None:
        return None
    # temperature to degC if it looks like Kelvin
    t_C = t - 273.15 if np.nanmean(t) > 100 else t
    return t_C, u, v, lats, lons, valid, run

def temperature_advection(t_C, u, v, lons, lats):
    """Return horizontal temperature advection in °C/hour.
    adv = -(u dT/dx + v dT/dy), with grid spacing in metres."""
    R = 6371000.0
    latr = np.deg2rad(lats)
    dlon = np.deg2rad(np.gradient(lons))
    dlat = np.deg2rad(np.gradient(lats))
    # distance per grid step (m): dx varies with latitude, dy constant
    dx = R * np.cos(latr)[:, None] * dlon[None, :]
    dy = (R * dlat)[:, None] * np.ones((1, len(lons)))
    dTdy, dTdx = np.gradient(t_C)            # index order: [lat, lon]
    dTdx = dTdx / dx
    dTdy = dTdy / dy
    adv = -(u * dTdx + v * dTdy)             # °C per second
    return adv * 3600.0                      # -> °C per hour
##%%
# ---- (run once) inspect variables in an 850 file ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} "
                  f"shortName={a.get('GRIB_shortName')} "
                  f"cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- advection colormap (°C/h): blue=cold advection, red=warm advection ----
adv_levels = [-3, -2, -1.5, -1, -0.5, -0.2, 0.2, 0.5, 1, 1.5, 2, 3]
adv_cmap = plt.get_cmap('RdBu_r')
adv_norm = mcolors.BoundaryNorm(adv_levels, adv_cmap.N)
##%%
# ---- plot: advection shaded + temperature contours + streamlines ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

def add_iran_border(ax):
    # other countries thin grey
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    # sea coastline (Persian Gulf, Gulf of Oman, etc.) at 1.5 px
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    # Caspian Sea coastline at 1.5 px (it is a lake, drawn separately)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='#eef4fb',
                          edgecolor='black', linewidth=1.5, zorder=4,
                          joinstyle='round', capstyle='round')
    # Iran provinces: 1.2 px
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.2, zorder=5,
                          joinstyle='round', capstyle='round')
    # Iran national border: 1.8 px, on top
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none',
                      edgecolor='black', linewidth=1.8, zorder=6,
                      joinstyle='round', capstyle='round')

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : need t + u + v, not all found, skipping')
        continue
    t_C, u, v, lats, lons, valid, run = rec
    adv = temperature_advection(t_C, u, v, lons, lats)
    u_kt, v_kt = u * MS_TO_KT, v * MS_TO_KT

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # 1) shaded temperature advection (°C/h)
    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, adv, levels=adv_levels, cmap=adv_cmap,
                     norm=adv_norm, extend='both', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=adv_levels)
    cbar.set_label(CREDIT)

    # 2) temperature contours (°C), coloured differently (dark red), labelled
    lo = np.floor(np.nanmin(t_C) / TEMP_CONTOUR_STEP) * TEMP_CONTOUR_STEP
    hi = np.ceil(np.nanmax(t_C) / TEMP_CONTOUR_STEP) * TEMP_CONTOUR_STEP
    tlevels = np.arange(lo, hi + TEMP_CONTOUR_STEP, TEMP_CONTOUR_STEP)
    cs = ax.contour(lons, lats, t_C, levels=tlevels, colors='#b30000',
                    linewidths=0.9, linestyles='dotted', transform=ccrs.PlateCarree())
    ax.clabel(cs, fmt='%d', fontsize=7, inline=True)

    # 3) streamlines (flip if lat runs N->S)
    lon1d, lat1d, U, V = lons, lats, u_kt, v_kt
    if lat1d[0] > lat1d[-1]:
        lat1d = lat1d[::-1]; U = U[::-1, :]; V = V[::-1, :]
    ax.streamplot(lon1d, lat1d, U, V, density=STREAM_DENSITY,
                  linewidth=0.6, color='black', arrowsize=0.8,
                  transform=ccrs.PlateCarree())

    add_iran_border(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded) + Temp Contours (°C) + Streamlines\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=11)

    out = os.path.join(OUTPUT_DIR, f'tempadv_stream_{LEVEL_HPA}_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (adv {np.nanmin(adv):.2f}..{np.nanmax(adv):.2f} {PARAM_UNIT})')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, f'tempadv_stream_{LEVEL_HPA}.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 700 hPa Relative Humidity (shaded) + Wind Streamlines

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 700
PRODUCT   = 'RH_Streamlines_700'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '700 hPa Relative Humidity'
PARAM_UNIT = '%'

MS_TO_KT = 1.94384
STREAM_DENSITY = 2.5

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m',
                                   category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   '
      f'Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.5, zorder=4,
                          joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.2, zorder=5,
                          joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none',
                      edgecolor='black', linewidth=1.8, zorder=6,
                      joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    """Return (rh, u, v, lats, lons, valid, run): RH in %, wind m/s.
    RH by paramId 157 ('r') or CF relative_humidity; U/V by 131/132 or CF name."""
    rh = u = v = w = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 157 or sn == 'r' or cf == 'relative_humidity') and rh is None:
                rh = np.asarray(da.values)
            elif (pid == 135 or sn == 'w' or cf == 'lagrangian_tendency_of_air_pressure') and w is None:
                w = np.asarray(da.values)
    if u is None or v is None or rh is None:
        return None
    return rh, w, u, v, lats, lons, valid, run
##%%
# ---- (run once) inspect variables in a 700 hPa file ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} "
                  f"shortName={a.get('GRIB_shortName')} "
                  f"cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- relative humidity colormap (%) ----
rh_levels = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
rh_colors = ['#b06a2c','#c98a4b','#dcae74','#ecd2a4','#f4eccf','#e6f0d8',
             '#bfe0b0','#86c87f','#49ac5e','#1f8f5f']
rh_cmap = mcolors.ListedColormap(rh_colors)
rh_norm = mcolors.BoundaryNorm(rh_levels, rh_cmap.N)
##%%
# ---- plot 700 hPa RH (shaded) + streamlines ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : need r + u + v, not all found, skipping')
        continue
    rh, w, u, v, lats, lons, valid, run = rec
    rh = np.clip(rh, 0, 100)
    u_kt, v_kt = u * MS_TO_KT, v * MS_TO_KT

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    cf = ax.contourf(lons, lats, rh, levels=rh_levels, cmap=rh_cmap,
                     norm=rh_norm, extend='neither', transform=ccrs.PlateCarree())
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=rh_levels)
    cbar.set_label(CREDIT)

    lon1d, lat1d, U, V = lons, lats, u_kt, v_kt
    if lat1d[0] > lat1d[-1]:
        lat1d = lat1d[::-1]; U = U[::-1, :]; V = V[::-1, :]
    ax.streamplot(lon1d, lat1d, U, V, density=STREAM_DENSITY,
                  linewidth=0.6, color='black', arrowsize=0.8,
                  transform=ccrs.PlateCarree())

    # vertical velocity (omega, Pa/s): dotted; blue ascent (w<0), red descent (w>0)
    if w is not None:
        cw_up = ax.contour(lons, lats, w, levels=[-2.0,-1.5,-1.0,-0.5,-0.2], colors='blue',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        cw_dn = ax.contour(lons, lats, w, levels=[0.2,0.5,1.0,1.5,2.0], colors='red',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        ax.clabel(cw_up, fmt='%.1f', fontsize=6, inline=True)
        ax.clabel(cw_dn, fmt='%.1f', fontsize=6, inline=True)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded) + Streamlines + VV (dotted)\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'rh_stream_{LEVEL_HPA}_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (RH {np.nanmin(rh):.0f}-{np.nanmax(rh):.0f} %)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, f'rh_stream_{LEVEL_HPA}.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 600 hPa Relative Humidity (shaded) + Wind Streamlines

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 600
PRODUCT   = 'RH_Streamlines_600'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '600 hPa Relative Humidity'
PARAM_UNIT = '%'

MS_TO_KT = 1.94384
STREAM_DENSITY = 2.5

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m',
                                   category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   '
      f'Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.5, zorder=4,
                          joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.2, zorder=5,
                          joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none',
                      edgecolor='black', linewidth=1.8, zorder=6,
                      joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    """Return (rh, u, v, lats, lons, valid, run): RH in %, wind m/s.
    RH by paramId 157 ('r') or CF relative_humidity; U/V by 131/132 or CF name."""
    rh = u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 157 or sn == 'r' or cf == 'relative_humidity') and rh is None:
                rh = np.asarray(da.values)
    if u is None or v is None or rh is None:
        return None
    return rh, u, v, lats, lons, valid, run
##%%
# ---- (run once) inspect variables in a 600 hPa file ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} "
                  f"shortName={a.get('GRIB_shortName')} "
                  f"cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- relative humidity colormap (%) ----
rh_levels = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
rh_colors = ['#b06a2c','#c98a4b','#dcae74','#ecd2a4','#f4eccf','#e6f0d8',
             '#bfe0b0','#86c87f','#49ac5e','#1f8f5f']
rh_cmap = mcolors.ListedColormap(rh_colors)
rh_norm = mcolors.BoundaryNorm(rh_levels, rh_cmap.N)
##%%
# ---- plot 600 hPa RH (shaded) + streamlines ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : need r + u + v, not all found, skipping')
        continue
    rh, u, v, lats, lons, valid, run = rec
    rh = np.clip(rh, 0, 100)
    u_kt, v_kt = u * MS_TO_KT, v * MS_TO_KT

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    if np.ndim(lons) == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = lons, lats
    cf = ax.contourf(lon2d, lat2d, rh, levels=rh_levels, cmap=rh_cmap,
                     norm=rh_norm, extend='neither', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=rh_levels)
    cbar.set_label(CREDIT)

    lon1d, lat1d, U, V = lons, lats, u_kt, v_kt
    if lat1d[0] > lat1d[-1]:
        lat1d = lat1d[::-1]; U = U[::-1, :]; V = V[::-1, :]
    ax.streamplot(lon1d, lat1d, U, V, density=STREAM_DENSITY,
                  linewidth=0.6, color='black', arrowsize=0.8,
                  transform=ccrs.PlateCarree())

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded) + Streamlines\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'rh_stream_{LEVEL_HPA}_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (RH {np.nanmin(rh):.0f}-{np.nanmax(rh):.0f} %)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, f'rh_stream_{LEVEL_HPA}.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 500 hPa Relative Vorticity (shaded) + Height contours + Temperature contours

In [ ]:
##%% md
# JMA GSM — 500 hPa Relative Vorticity (shaded) + Height contours + Temperature contours

import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
from scipy.ndimage import gaussian_filter
import json
from matplotlib.ticker import FuncFormatter
##%%
# ---- CONFIG ----  [HEIGHT-LABEL BUILD v2: dkm + auto+manual labels]
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 500
PRODUCT   = 'Vorticity_500'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '500 hPa Relative Vorticity'
PARAM_UNIT = '×10⁻⁵ s⁻¹'

HEIGHT_STEP = 30   # gpm (= 3 dkm) between height contours at 500
TEMP_STEP   = 2

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    # gh (gpm), t (degC), u, v (m/s); detected by paramId or CF name
    gh = t = u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 156 or sn == 'gh' or cf == 'geopotential_height') and gh is None:
                gh = np.asarray(da.values)
            elif (pid == 129 or sn == 'z' or cf == 'geopotential') and gh is None:
                gh = np.asarray(da.values) / 9.80665
            elif (pid == 130 or sn == 't' or cf == 'air_temperature') and t is None:
                t = np.asarray(da.values)
    if u is None or v is None:
        return None
    t_C = None
    if t is not None:
        t_C = t - 273.15 if np.nanmean(t) > 100 else t
    return gh, t_C, u, v, lats, lons, valid, run

def relative_vorticity(u, v, lons, lats):
    # zeta = dv/dx - du/dy (s^-1), metre spacing; dx shrinks with latitude
    R = 6371000.0
    latr = np.deg2rad(lats)
    dlon = np.deg2rad(np.gradient(lons))
    dlat = np.deg2rad(np.gradient(lats))
    dx = R * np.cos(latr)[:, None] * dlon[None, :]
    dy = (R * dlat)[:, None] * np.ones((1, len(lons)))
    dudy = np.gradient(u, axis=0) / dy
    dvdx = np.gradient(v, axis=1) / dx
    return dvdx - dudy
##%%
# ---- (run once) inspect variables in a 500 hPa file ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} shortName={a.get('GRIB_shortName')} cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- vorticity colormap (x10^-5 s^-1) ----
vort_levels = [-20, -15, -10, -7, -5, -3, -1, 1, 3, 5, 7, 10, 15, 20]
vort_cmap = plt.get_cmap('RdBu_r')
vort_norm = mcolors.BoundaryNorm(vort_levels, vort_cmap.N)
##%%
# ---- plot ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : U/V not found, skipping')
        continue
    gh, t_C, u, v, lats, lons, valid, run = rec
    zeta = relative_vorticity(u, v, lons, lats) * 1e5

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # light smoothing for a clean look on the 0.5deg grid, then contourf.
    # transform_first=True avoids the cartopy/shapely 'GeometryCollection'
    # contourf bug while keeping smooth filled bands.
    zeta_s = gaussian_filter(np.nan_to_num(zeta), sigma=1.0)
    lon2d, lat2d = np.meshgrid(lons, lats)   # 2-D coords for transform_first
    cf = ax.contourf(lon2d, lat2d, zeta_s, levels=vort_levels, cmap=vort_cmap,
                     norm=vort_norm, extend='both', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50, ticks=vort_levels)
    cbar.set_label(CREDIT)

    if gh is not None:
        lo = np.floor(np.nanmin(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hi = np.ceil(np.nanmax(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hlev = np.arange(lo, hi+HEIGHT_STEP, HEIGHT_STEP)
        # Convert to DECAMETERS first, then contour, so labels are plain integers
        # using the same '%d' mechanism that already works for the temperature lines.
        gh_dkm = gh / 10.0
        hlev_dkm = hlev / 10.0
        csh = ax.contour(lon2d, lat2d, gh_dkm, levels=hlev_dkm, colors='black', linewidths=1.2,
                         transform=ccrs.PlateCarree(), transform_first=True)
        # Label every height contour. Try auto inline placement; if a contour gets no
        # label, force one manually near the map-centre longitude so NO line is unlabeled.
        lbls = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, inline_spacing=2)
        for _t in lbls:
            _t.set_zorder(12)
        # belt-and-suspenders: manual labels at the centre meridian for any line that
        # the auto-placer skipped (long near-horizontal contours sometimes get skipped).
        try:
            cx = 0.5 * (lon_min + lon_max)
            ci = int(round(np.interp(cx, lons if np.ndim(lons)==1 else lons[0], np.arange(len(lons if np.ndim(lons)==1 else lons[0])))))
            ghcol = (gh[:, ci] / 10.0)
            latcol = lats if np.ndim(lats)==1 else lats[:, 0]
            man_pts = []
            for lv in hlev_dkm:
                d = ghcol - lv
                sgn = np.sign(d)
                xs = np.where(np.diff(sgn) != 0)[0]
                if len(xs):
                    k = xs[len(xs)//2]
                    if lat_min <= latcol[k] <= lat_max:
                        man_pts.append((cx, float(latcol[k])))
            if man_pts:
                ml = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, manual=man_pts)
                for _t in ml:
                    _t.set_zorder(12)
        except Exception as _e:
            print('manual height-label fallback skipped:', _e)

    if t_C is not None:
        lo = np.floor(np.nanmin(t_C)/TEMP_STEP)*TEMP_STEP
        hi = np.ceil(np.nanmax(t_C)/TEMP_STEP)*TEMP_STEP
        tlev = np.arange(lo, hi+TEMP_STEP, TEMP_STEP)
        cst = ax.contour(lons, lats, t_C, levels=tlev, colors='green', linewidths=1.0,
                         linestyles='dashed', transform=ccrs.PlateCarree())
        ax.clabel(cst, fmt='%d', fontsize=6, inline=True)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded)\n'
        f'+ Height contours (dkm) + Temperature (°C, green)\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    Forecast Hour: +{hour:03d}h',
        fontsize=11)

    out = os.path.join(OUTPUT_DIR, f'vort500_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (zeta {np.nanmin(zeta):.1f}..{np.nanmax(zeta):.1f})')
##%%
# ---- build GIF ----
import imageio.v2 as imageio
imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'vorticity_500.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 300 hPa Wind Speed (shaded) + Height contours + Vertical Velocity contours

In [ ]:
# JMA GSM — 300 hPa Wind Speed (shaded) + Height contours + Vertical Velocity contours
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
from scipy.ndimage import gaussian_filter
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
from matplotlib.ticker import FuncFormatter
##%%
# ---- CONFIG ----  [HEIGHT-LABEL BUILD v2: dkm + auto+manual labels]
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 300
PRODUCT   = 'Wind_VV_300'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '300 hPa Wind Speed'
PARAM_UNIT = 'kt'

MS_TO_KT = 1.94384
HEIGHT_STEP = 60   # gpm (= 6 dkm) between height contours

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    # gh (gpm), omega (Pa/s), u, v (m/s)
    gh = w = u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 156 or sn == 'gh' or cf == 'geopotential_height') and gh is None:
                gh = np.asarray(da.values)
            elif (pid == 129 or sn == 'z' or cf == 'geopotential') and gh is None:
                gh = np.asarray(da.values) / 9.80665
            elif (pid == 135 or sn == 'w' or cf == 'lagrangian_tendency_of_air_pressure') and w is None:
                w = np.asarray(da.values)
    if u is None or v is None:
        return None
    return gh, w, u, v, lats, lons, valid, run
##%%
# ---- (run once) inspect variables ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} shortName={a.get('GRIB_shortName')} cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- wind speed colormap (knots), upper-level scaling ----
wind_levels = [20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160]
wind_colors = ['#d8eaf7','#b6d6ef','#8fbce4','#6a9fd8','#5aa0f0','#21d07a',
               '#ffe14d','#ffac1c','#ff5e1c','#e01010','#8a00b3']
wind_cmap = mcolors.ListedColormap(wind_colors)
wind_cmap.set_under('#ffffff')
wind_cmap.set_over('#4a0072')
wind_norm = mcolors.BoundaryNorm(wind_levels, wind_cmap.N)
##%%
# ---- plot ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : U/V not found, skipping')
        continue
    gh, w, u, v, lats, lons, valid, run = rec
    speed_kt = np.sqrt(u**2 + v**2) * MS_TO_KT
    lon2d, lat2d = np.meshgrid(lons, lats)

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # 1) shaded wind speed (smoothed contourf; transform_first avoids cartopy bug)
    sp_s = gaussian_filter(np.nan_to_num(speed_kt), sigma=0.8)
    cf = ax.contourf(lon2d, lat2d, sp_s, levels=wind_levels, cmap=wind_cmap, norm=wind_norm,
                     extend='max', transform=ccrs.PlateCarree(), transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50, ticks=wind_levels)
    cbar.set_label(CREDIT)

    # 2) geopotential height: solid black contours
    if gh is not None:
        lo = np.floor(np.nanmin(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hi = np.ceil(np.nanmax(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hlev = np.arange(lo, hi+HEIGHT_STEP, HEIGHT_STEP)
        # Convert to DECAMETERS first, then contour, so labels are plain integers
        # using the same '%d' mechanism that already works for the temperature lines.
        gh_dkm = gh / 10.0
        hlev_dkm = hlev / 10.0
        csh = ax.contour(lon2d, lat2d, gh_dkm, levels=hlev_dkm, colors='black', linewidths=1.2,
                         transform=ccrs.PlateCarree(), transform_first=True)
        # Label every height contour. Try auto inline placement; if a contour gets no
        # label, force one manually near the map-centre longitude so NO line is unlabeled.
        lbls = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, inline_spacing=2)
        for _t in lbls:
            _t.set_zorder(12)
        # belt-and-suspenders: manual labels at the centre meridian for any line that
        # the auto-placer skipped (long near-horizontal contours sometimes get skipped).
        try:
            cx = 0.5 * (lon_min + lon_max)
            ci = int(round(np.interp(cx, lons if np.ndim(lons)==1 else lons[0], np.arange(len(lons if np.ndim(lons)==1 else lons[0])))))
            ghcol = (gh[:, ci] / 10.0)
            latcol = lats if np.ndim(lats)==1 else lats[:, 0]
            man_pts = []
            for lv in hlev_dkm:
                d = ghcol - lv
                sgn = np.sign(d)
                xs = np.where(np.diff(sgn) != 0)[0]
                if len(xs):
                    k = xs[len(xs)//2]
                    if lat_min <= latcol[k] <= lat_max:
                        man_pts.append((cx, float(latcol[k])))
            if man_pts:
                ml = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, manual=man_pts)
                for _t in ml:
                    _t.set_zorder(12)
        except Exception as _e:
            print('manual height-label fallback skipped:', _e)

    # 3) vertical velocity (omega, Pa/s): blue dashed ascent (w<0), red solid descent (w>0)
    if w is not None:
        wlev_up = [-2.0, -1.5, -1.0, -0.5, -0.2]
        wlev_dn = [0.2, 0.5, 1.0, 1.5, 2.0]
        cw_up = ax.contour(lons, lats, w, levels=wlev_up, colors='blue',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        cw_dn = ax.contour(lons, lats, w, levels=wlev_dn, colors='red',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        ax.clabel(cw_up, fmt='%.1f', fontsize=6, inline=True)
        ax.clabel(cw_dn, fmt='%.1f', fontsize=6, inline=True)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded)\n'
        f'+ Height contours (dkm) + Vertical Velocity (ω, Pa/s, dotted)\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    Forecast Hour: +{hour:03d}h',
        fontsize=11)

    out = os.path.join(OUTPUT_DIR, f'wind_vv_{LEVEL_HPA}_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (max wind {np.nanmax(speed_kt):.0f} kt)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio
imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'wind_vv_300.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — 200 hPa Wind Speed (shaded) + Height contours + Vertical Velocity contours

In [ ]:
##%% md
# JMA GSM — 200 hPa Wind Speed (shaded) + Height contours + Vertical Velocity contours

import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
from scipy.ndimage import gaussian_filter
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
from matplotlib.ticker import FuncFormatter
##%%
# ---- CONFIG ----  [HEIGHT-LABEL BUILD v2: dkm + auto+manual labels]
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 200
PRODUCT   = 'Wind_VV_200'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '200 hPa Wind Speed'
PARAM_UNIT = 'kt'

MS_TO_KT = 1.94384
HEIGHT_STEP = 60   # gpm (= 6 dkm) between height contours

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 23, 42
lon_min, lon_max = 40, 65

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    # gh (gpm), omega (Pa/s), u, v (m/s)
    gh = w = u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 156 or sn == 'gh' or cf == 'geopotential_height') and gh is None:
                gh = np.asarray(da.values)
            elif (pid == 129 or sn == 'z' or cf == 'geopotential') and gh is None:
                gh = np.asarray(da.values) / 9.80665
            elif (pid == 135 or sn == 'w' or cf == 'lagrangian_tendency_of_air_pressure') and w is None:
                w = np.asarray(da.values)
    if u is None or v is None:
        return None
    return gh, w, u, v, lats, lons, valid, run
##%%
# ---- (run once) inspect variables ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} shortName={a.get('GRIB_shortName')} cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- wind speed colormap (knots), upper-level scaling ----
wind_levels = [20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160]
wind_colors = ['#d8eaf7','#b6d6ef','#8fbce4','#6a9fd8','#5aa0f0','#21d07a',
               '#ffe14d','#ffac1c','#ff5e1c','#e01010','#8a00b3']
wind_cmap = mcolors.ListedColormap(wind_colors)
wind_cmap.set_under('#ffffff')
wind_cmap.set_over('#4a0072')
wind_norm = mcolors.BoundaryNorm(wind_levels, wind_cmap.N)
##%%
# ---- plot ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : U/V not found, skipping')
        continue
    gh, w, u, v, lats, lons, valid, run = rec
    speed_kt = np.sqrt(u**2 + v**2) * MS_TO_KT
    lon2d, lat2d = np.meshgrid(lons, lats)

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # 1) shaded wind speed (smoothed contourf; transform_first avoids cartopy bug)
    sp_s = gaussian_filter(np.nan_to_num(speed_kt), sigma=0.8)
    cf = ax.contourf(lon2d, lat2d, sp_s, levels=wind_levels, cmap=wind_cmap, norm=wind_norm,
                     extend='max', transform=ccrs.PlateCarree(), transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50, ticks=wind_levels)
    cbar.set_label(CREDIT)

    # 2) geopotential height: solid black contours
    if gh is not None:
        lo = np.floor(np.nanmin(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hi = np.ceil(np.nanmax(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hlev = np.arange(lo, hi+HEIGHT_STEP, HEIGHT_STEP)
        # Convert to DECAMETERS first, then contour, so labels are plain integers
        # using the same '%d' mechanism that already works for the temperature lines.
        gh_dkm = gh / 10.0
        hlev_dkm = hlev / 10.0
        csh = ax.contour(lon2d, lat2d, gh_dkm, levels=hlev_dkm, colors='black', linewidths=1.2,
                         transform=ccrs.PlateCarree(), transform_first=True)
        # Label every height contour. Try auto inline placement; if a contour gets no
        # label, force one manually near the map-centre longitude so NO line is unlabeled.
        lbls = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, inline_spacing=2)
        for _t in lbls:
            _t.set_zorder(12)
        # belt-and-suspenders: manual labels at the centre meridian for any line that
        # the auto-placer skipped (long near-horizontal contours sometimes get skipped).
        try:
            cx = 0.5 * (lon_min + lon_max)
            ci = int(round(np.interp(cx, lons if np.ndim(lons)==1 else lons[0], np.arange(len(lons if np.ndim(lons)==1 else lons[0])))))
            ghcol = (gh[:, ci] / 10.0)
            latcol = lats if np.ndim(lats)==1 else lats[:, 0]
            man_pts = []
            for lv in hlev_dkm:
                d = ghcol - lv
                sgn = np.sign(d)
                xs = np.where(np.diff(sgn) != 0)[0]
                if len(xs):
                    k = xs[len(xs)//2]
                    if lat_min <= latcol[k] <= lat_max:
                        man_pts.append((cx, float(latcol[k])))
            if man_pts:
                ml = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, manual=man_pts)
                for _t in ml:
                    _t.set_zorder(12)
        except Exception as _e:
            print('manual height-label fallback skipped:', _e)

    # 3) vertical velocity (omega, Pa/s): blue dashed ascent (w<0), red solid descent (w>0)
    if w is not None:
        wlev_up = [-2.0, -1.5, -1.0, -0.5, -0.2]
        wlev_dn = [0.2, 0.5, 1.0, 1.5, 2.0]
        cw_up = ax.contour(lons, lats, w, levels=wlev_up, colors='blue',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        cw_dn = ax.contour(lons, lats, w, levels=wlev_dn, colors='red',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        ax.clabel(cw_up, fmt='%.1f', fontsize=6, inline=True)
        ax.clabel(cw_dn, fmt='%.1f', fontsize=6, inline=True)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded)\n'
        f'+ Height contours (dkm) + Vertical Velocity (ω, Pa/s, dotted)\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    Forecast Hour: +{hour:03d}h',
        fontsize=11)

    out = os.path.join(OUTPUT_DIR, f'wind_vv_{LEVEL_HPA}_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (max wind {np.nanmax(speed_kt):.0f} kt)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio
imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'wind_vv_200.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — Persian Gulf & Gulf of Oman: Wind arrows + estimated wave height

In [ ]:
import os, glob, warnings, json
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import matplotlib.path as mpath
from shapely.geometry import Point
from shapely.prepared import prep
from shapely.ops import unary_union
##%%
# ---- CONFIG ----
LEVEL   = 'surface'
PRODUCT = 'Gulf_Wind_Waves'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = 'Estimated Wave Height'
PARAM_UNIT = 'm'

MS_TO_KT = 1.94384
WAVE_COEF = 0.016           # Hs ~ WAVE_COEF * U^2 (fetch-limited Gulf; gentler than open-ocean 0.0246)
BARB_STRIDE = 3            # plot every Nth wind barb (lower = denser)
STREAM_DENSITY = 2.5       # streamline density (higher = more lines)

CREDIT = ('Based on JMA GSM data (0.25\u00B0 surface). Wave height is an ESTIMATE from '
          '10 m wind (Hs\u22480.016\u00B7U\u00B2 fetch-limited), not a wave model.\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

# Persian Gulf + Gulf of Oman zoom box
lat_min, lat_max = 22, 31
lon_min, lon_max = 46, 62

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level)=='surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m: continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand: return None
    cand.sort(); return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f: d=json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r: return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — download surface for this run?')
##%%
# ---- borders + a land mask so wave/arrows show over SEA ONLY ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')
_land  = shapereader.natural_earth(resolution='10m', category='physical', name='land')

def _simplify(geoms, tol):
    out=[]
    for g in geoms:
        try: out.append(g.simplify(tol, preserve_topology=True))
        except Exception: out.append(g)
    return out

def _iran_country_geoms():
    g=[]
    for rec in shapereader.Reader(_countries).records():
        nm=(rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm=(rec.attributes.get('ADMIN') or '')
        if 'Iran' in nm or 'Iran' in adm: g.append(rec.geometry)
    return g

def _iran_province_geoms():
    g=[]
    for rec in shapereader.Reader(_provinces).records():
        at=rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            g.append(rec.geometry)
    return g

def _caspian_geom():
    g=[]
    for rec in shapereader.Reader(_lakes).records():
        nm=str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm: g.append(rec.geometry)
    return g

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), 0.03)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), 0.06)
CASPIAN        = _simplify(_caspian_geom(), 0.03)

# union of all land polygons (for masking points that fall on land)
_land_geoms = [rec.geometry for rec in shapereader.Reader(_land).records()]
LAND_UNION = prep(unary_union(_land_geoms))
print('Land polygons loaded for sea mask:', len(_land_geoms))

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical','coastline','10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=6)
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.0, zorder=6, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=7, joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out=[]
    for f in files:
        try:
            fd=os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort(); return out

def read_uv(path):
    u=v=None; lats=lons=valid=run=None
    for ds in cfgrib.open_datasets(path):
        for vname, da in ds.data_vars.items():
            pid=da.attrs.get('GRIB_paramId')
            if pid==165 and u is None:
                u=np.asarray(da.values)
                lats=ds['latitude'].values; lons=ds['longitude'].values
                valid=da['valid_time'].values if 'valid_time' in da.coords else ''
                run=da['time'].values if 'time' in da.coords else ''
            elif pid==166 and v is None:
                v=np.asarray(da.values)
    if u is None or v is None: return None
    return u, v, lats, lons, valid, run

def fmt_utc(t):
    try:
        dt=np.datetime64(t,'h'); return f"{str(dt).replace('T',' ')}:00 UTC"
    except Exception:
        return str(t)

_SEA_MASK_CACHE = {}
def sea_mask(lons2d, lats2d):
    """Boolean array True over sea (not on land). Cached by grid shape+extent."""
    key=(lons2d.shape, float(lons2d.min()), float(lons2d.max()), float(lats2d.min()), float(lats2d.max()))
    if key in _SEA_MASK_CACHE:
        return _SEA_MASK_CACHE[key]
    sea=np.ones(lons2d.shape, dtype=bool)
    for j in range(lons2d.shape[0]):
        for i in range(lons2d.shape[1]):
            if LAND_UNION.contains(Point(float(lons2d[j,i]), float(lats2d[j,i]))):
                sea[j,i]=False
    _SEA_MASK_CACHE[key]=sea
    return sea
##%%
# ---- wave height colormap (m) ----
wave_levels = [0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
wave_colors = ['#dcefff','#a8d4f5','#6fb0e8','#3f86cf','#7ec46a','#ddd23f',
               '#f4a23a','#ef6a2e','#df2f2f','#9e1530','#641a8c']
wave_cmap = mcolors.ListedColormap(wave_colors)
wave_cmap.set_under('#ffffff00'); wave_cmap.set_over('#3a0030')
wave_norm = mcolors.BoundaryNorm(wave_levels, wave_cmap.N)
##%%
# ---- plot: estimated wave height (sea only) + wind arrows over the Gulf/Oman ----
if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
    print(f'{len(files)} surface files found')

for hour, path in files:
    rec = read_uv(path)
    if rec is None:
        print(f'+{hour:03d}h : 10 m wind not found, skipping'); continue
    u, v, lats, lons, valid, run = rec
    spd = np.sqrt(u**2 + v**2)                 # m/s
    hs = WAVE_COEF * spd**2                     # estimated Hs (m)

    lon2d, lat2d = np.meshgrid(lons, lats)
    sea = sea_mask(lon2d, lat2d)
    hs_sea = np.where(sea, hs, np.nan)          # show waves over sea only

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(12,9), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#efe7da', zorder=1)
    ax.add_feature(cfeature.OCEAN, facecolor='#eaf4fb', zorder=0)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # shaded estimated wave height (sea only); transform_first avoids cartopy bug
    cf = ax.contourf(lon2d, lat2d, hs_sea, levels=wave_levels, cmap=wave_cmap, norm=wave_norm,
                     extend='both', transform=ccrs.PlateCarree(), transform_first=True, zorder=2)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.06, aspect=50, ticks=wave_levels)
    cbar.set_label(CREDIT)

    # standard wind BARBS over the sea only (direction + speed in knots).
    st = BARB_STRIDE
    u_kt = u * MS_TO_KT
    v_kt = v * MS_TO_KT
    ub = np.where(sea, u_kt, np.nan)[::st, ::st]
    vb = np.where(sea, v_kt, np.nan)[::st, ::st]
    bx = lon2d[::st, ::st]; by = lat2d[::st, ::st]
    ax.barbs(bx, by, ub, vb, transform=ccrs.PlateCarree(), zorder=5,
             length=5.5, linewidth=0.6, color='black',
             barbcolor='black', flagcolor='black',
             sizes=dict(emptybarb=0.0))

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}, estimated) + 10 m Wind barbs (kt) — Persian Gulf & Gulf of Oman\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    Forecast Hour: +{hour:03d}h',
        fontsize=10)

    out = os.path.join(OUTPUT_DIR, f'gulf_wind_wave_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    mx = np.nanmax(hs_sea) if np.isfinite(hs_sea).any() else 0.0
    print(f'+{hour:03d}h : saved {out}  (max est Hs {mx:.1f} m)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio
imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'gulf_wind_wave.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

# JMA GSM — Caspian Sea: Wind arrows + estimated wave height

In [ ]:
import os, glob, warnings, json
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import matplotlib.path as mpath
from shapely.geometry import Point
from shapely.prepared import prep
from shapely.ops import unary_union
##%%
# ---- CONFIG ----
LEVEL   = 'surface'
PRODUCT = 'Caspian_Wind_Waves'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = 'Estimated Wave Height'
PARAM_UNIT = 'm'

MS_TO_KT = 1.94384
WAVE_COEF = 0.016           # Hs ~ WAVE_COEF * U^2 (fetch-limited Gulf; gentler than open-ocean 0.0246)
BARB_STRIDE = 3            # plot every Nth wind barb (lower = denser)
STREAM_DENSITY = 2.5       # streamline density (higher = more lines)

CREDIT = ('Based on JMA GSM data (0.25\u00B0 surface). Wave height is an ESTIMATE from '
          '10 m wind (Hs\u22480.016\u00B7U\u00B2 fetch-limited), not a wave model.\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

# Caspian Sea zoom box
lat_min, lat_max = 36, 47.5
lon_min, lon_max = 46, 55

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level)=='surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m: continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand: return None
    cand.sort(); return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f: d=json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL)
    if r: return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, LEVEL)
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — download surface for this run?')
##%%
# ---- borders + a land mask so wave/arrows show over SEA ONLY ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')
_land  = shapereader.natural_earth(resolution='10m', category='physical', name='land')

def _simplify(geoms, tol):
    out=[]
    for g in geoms:
        try: out.append(g.simplify(tol, preserve_topology=True))
        except Exception: out.append(g)
    return out

def _iran_country_geoms():
    g=[]
    for rec in shapereader.Reader(_countries).records():
        nm=(rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm=(rec.attributes.get('ADMIN') or '')
        if 'Iran' in nm or 'Iran' in adm: g.append(rec.geometry)
    return g

def _iran_province_geoms():
    g=[]
    for rec in shapereader.Reader(_provinces).records():
        at=rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            g.append(rec.geometry)
    return g

def _caspian_geom():
    g=[]
    for rec in shapereader.Reader(_lakes).records():
        nm=str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm: g.append(rec.geometry)
    return g

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), 0.03)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), 0.06)
CASPIAN        = _simplify(_caspian_geom(), 0.03)

# union of all land polygons (for masking points that fall on land)
_land_geoms = [rec.geometry for rec in shapereader.Reader(_land).records()]
LAND_UNION = prep(unary_union(_land_geoms))
# Caspian water polygons (so the inland sea counts as SEA even if 'land' covers it)
_casp_water = []
for rec in shapereader.Reader(_lakes).records():
    nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
    if 'Caspian' in nm:
        _casp_water.append(rec.geometry)
CASPIAN_WATER = prep(unary_union(_casp_water)) if _casp_water else None
print('Land polygons loaded for sea mask:', len(_land_geoms),
      '| Caspian water polys:', len(_casp_water))

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical','coastline','10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=6)
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.0, zorder=6, joinstyle='round', capstyle='round')
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=6, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=7, joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_surface_files(data_dir):
    files = glob.glob(os.path.join(data_dir, '*Lsurf_FD*_grib2.bin'))
    out=[]
    for f in files:
        try:
            fd=os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort(); return out

def read_uv(path):
    u=v=None; lats=lons=valid=run=None
    for ds in cfgrib.open_datasets(path):
        for vname, da in ds.data_vars.items():
            pid=da.attrs.get('GRIB_paramId')
            if pid==165 and u is None:
                u=np.asarray(da.values)
                lats=ds['latitude'].values; lons=ds['longitude'].values
                valid=da['valid_time'].values if 'valid_time' in da.coords else ''
                run=da['time'].values if 'time' in da.coords else ''
            elif pid==166 and v is None:
                v=np.asarray(da.values)
    if u is None or v is None: return None
    return u, v, lats, lons, valid, run

def fmt_utc(t):
    try:
        dt=np.datetime64(t,'h'); return f"{str(dt).replace('T',' ')}:00 UTC"
    except Exception:
        return str(t)

_SEA_MASK_CACHE = {}
def sea_mask(lons2d, lats2d):
    """Boolean array True over sea (not on land). Cached by grid shape+extent."""
    key=(lons2d.shape, float(lons2d.min()), float(lons2d.max()), float(lats2d.min()), float(lats2d.max()))
    if key in _SEA_MASK_CACHE:
        return _SEA_MASK_CACHE[key]
    sea=np.ones(lons2d.shape, dtype=bool)
    for j in range(lons2d.shape[0]):
        for i in range(lons2d.shape[1]):
            p = Point(float(lons2d[j,i]), float(lats2d[j,i]))
            if CASPIAN_WATER is not None and CASPIAN_WATER.contains(p):
                sea[j,i]=True       # Caspian water is sea
            elif LAND_UNION.contains(p):
                sea[j,i]=False
    _SEA_MASK_CACHE[key]=sea
    return sea
##%%
# ---- wave height colormap (m) ----
wave_levels = [0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
wave_colors = ['#dcefff','#a8d4f5','#6fb0e8','#3f86cf','#7ec46a','#ddd23f',
               '#f4a23a','#ef6a2e','#df2f2f','#9e1530','#641a8c']
wave_cmap = mcolors.ListedColormap(wave_colors)
wave_cmap.set_under('#ffffff00'); wave_cmap.set_over('#3a0030')
wave_norm = mcolors.BoundaryNorm(wave_levels, wave_cmap.N)
##%%
# ---- plot: estimated wave height (sea only) + wind arrows over the Gulf/Oman ----
if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_surface_files(DATA_DIR)
    print(f'{len(files)} surface files found')

for hour, path in files:
    rec = read_uv(path)
    if rec is None:
        print(f'+{hour:03d}h : 10 m wind not found, skipping'); continue
    u, v, lats, lons, valid, run = rec
    spd = np.sqrt(u**2 + v**2)                 # m/s
    hs = WAVE_COEF * spd**2                     # estimated Hs (m)

    lon2d, lat2d = np.meshgrid(lons, lats)
    sea = sea_mask(lon2d, lat2d)
    hs_sea = np.where(sea, hs, np.nan)          # show waves over sea only

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(12,9), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#efe7da', zorder=1)
    ax.add_feature(cfeature.OCEAN, facecolor='#eaf4fb', zorder=0)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # shaded estimated wave height (sea only); transform_first avoids cartopy bug
    cf = ax.contourf(lon2d, lat2d, hs_sea, levels=wave_levels, cmap=wave_cmap, norm=wave_norm,
                     extend='both', transform=ccrs.PlateCarree(), transform_first=True, zorder=2)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.06, aspect=50, ticks=wave_levels)
    cbar.set_label(CREDIT)

    # standard wind BARBS over the sea only (direction + speed in knots).
    st = BARB_STRIDE
    u_kt = u * MS_TO_KT
    v_kt = v * MS_TO_KT
    ub = np.where(sea, u_kt, np.nan)[::st, ::st]
    vb = np.where(sea, v_kt, np.nan)[::st, ::st]
    bx = lon2d[::st, ::st]; by = lat2d[::st, ::st]
    ax.barbs(bx, by, ub, vb, transform=ccrs.PlateCarree(), zorder=5,
             length=5.5, linewidth=0.6, color='black',
             barbcolor='black', flagcolor='black',
             sizes=dict(emptybarb=0.0))

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}, estimated) + 10 m Wind barbs (kt) — Caspian Sea\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    Forecast Hour: +{hour:03d}h',
        fontsize=10)

    out = os.path.join(OUTPUT_DIR, f'caspian_wind_wave_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    mx = np.nanmax(hs_sea) if np.isfinite(hs_sea).any() else 0.0
    print(f'+{hour:03d}h : saved {out}  (max est Hs {mx:.1f} m)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio
imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'caspian_wind_wave.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

## JMA GSM — 700 hPa Relative Humidity (shaded) + Wind Streamlines (WIDE)

In [ ]:
# JMA GSM — 700 hPa Relative Humidity (shaded) + Wind Streamlines (WIDE)
import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import json
##%%
# ---- CONFIG ----
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 700
PRODUCT   = 'RH_Streamlines_700_Wide'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '700 hPa Relative Humidity'
PARAM_UNIT = '%'

MS_TO_KT = 1.94384
STREAM_DENSITY = 2.5

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 5, 60
lon_min, lon_max = 30, 80

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders: countries, Iran provinces, Iran national border, sea coasts ----
_countries = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m',
                                       category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m',
                                   category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   '
      f'Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.5, zorder=4,
                          joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none',
                          edgecolor='black', linewidth=1.2, zorder=5,
                          joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none',
                      edgecolor='black', linewidth=1.8, zorder=6,
                      joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    """Return (rh, u, v, lats, lons, valid, run): RH in %, wind m/s.
    RH by paramId 157 ('r') or CF relative_humidity; U/V by 131/132 or CF name."""
    rh = u = v = w = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 157 or sn == 'r' or cf == 'relative_humidity') and rh is None:
                rh = np.asarray(da.values)
            elif (pid == 135 or sn == 'w' or cf == 'lagrangian_tendency_of_air_pressure') and w is None:
                w = np.asarray(da.values)
    if u is None or v is None or rh is None:
        return None
    return rh, w, u, v, lats, lons, valid, run
##%%
# ---- (run once) inspect variables in a 700 hPa file ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} "
                  f"shortName={a.get('GRIB_shortName')} "
                  f"cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- relative humidity colormap (%) ----
rh_levels = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
rh_colors = ['#b06a2c','#c98a4b','#dcae74','#ecd2a4','#f4eccf','#e6f0d8',
             '#bfe0b0','#86c87f','#49ac5e','#1f8f5f']
rh_cmap = mcolors.ListedColormap(rh_colors)
rh_norm = mcolors.BoundaryNorm(rh_levels, rh_cmap.N)
##%%
# ---- plot 700 hPa RH (shaded) + streamlines ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : need r + u + v, not all found, skipping')
        continue
    rh, w, u, v, lats, lons, valid, run = rec
    rh = np.clip(rh, 0, 100)
    u_kt, v_kt = u * MS_TO_KT, v * MS_TO_KT

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()},
                           figsize=(12, 10), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    cf = ax.contourf(lons, lats, rh, levels=rh_levels, cmap=rh_cmap,
                     norm=rh_norm, extend='neither', transform=ccrs.PlateCarree())
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50,
                        ticks=rh_levels)
    cbar.set_label(CREDIT)

    lon1d, lat1d, U, V = lons, lats, u_kt, v_kt
    if lat1d[0] > lat1d[-1]:
        lat1d = lat1d[::-1]; U = U[::-1, :]; V = V[::-1, :]
    ax.streamplot(lon1d, lat1d, U, V, density=STREAM_DENSITY,
                  linewidth=0.6, color='black', arrowsize=0.8,
                  transform=ccrs.PlateCarree())

    # vertical velocity (omega, Pa/s): dotted; blue ascent (w<0), red descent (w>0)
    if w is not None:
        cw_up = ax.contour(lons, lats, w, levels=[-2.0,-1.5,-1.0,-0.5,-0.2], colors='blue',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        cw_dn = ax.contour(lons, lats, w, levels=[0.2,0.5,1.0,1.5,2.0], colors='red',
                           linewidths=0.8, linestyles='dotted', transform=ccrs.PlateCarree())
        ax.clabel(cw_up, fmt='%.1f', fontsize=6, inline=True)
        ax.clabel(cw_dn, fmt='%.1f', fontsize=6, inline=True)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded) + Streamlines + VV (dotted)\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    '
        f'Forecast Hour: +{hour:03d}h',
        fontsize=12)

    out = os.path.join(OUTPUT_DIR, f'rh_stream_{LEVEL_HPA}_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (RH {np.nanmin(rh):.0f}-{np.nanmax(rh):.0f} %)')
##%%
# ---- build GIF ----
import imageio.v2 as imageio

imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, f'rh_stream_{LEVEL_HPA}.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)

## JMA GSM — 500 hPa Relative Vorticity (shaded) + Height contours + Temperature contours (WIDE)

In [ ]:
# JMA GSM — 500 hPa Relative Vorticity (shaded) + Height contours + Temperature contours (WIDE)

import os, glob, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import cfgrib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
from scipy.ndimage import gaussian_filter
import json
from matplotlib.ticker import FuncFormatter
##%%
# ---- CONFIG ----  [HEIGHT-LABEL BUILD v2: dkm + auto+manual labels]
# Finds the data automatically: pointer file (~/jma_latest_run.json) first,
# else scan DEFAULT_DATA_ROOT for the newest downloaded run.
LEVEL_HPA = 500
PRODUCT   = 'Vorticity_500_Wide'

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop', 'JMA-Data')

PARAM_NAME = '500 hPa Relative Vorticity'
PARAM_UNIT = '×10⁻⁵ s⁻¹'

HEIGHT_STEP = 30   # gpm (= 3 dkm) between height contours at 500
TEMP_STEP   = 2

CREDIT = ('Based on JMA GSM data (0.25\u00B0 resolution for surface layers; '
          '0.5\u00B0 for upper-air layers).\n'
          'Developed by H. Shokoohi & H. Mastaneh, Bushehr Meteorological Office, Iran.')

lat_min, lat_max = 5, 60
lon_min, lon_max = 30, 80

POINTER_PATH = os.path.join(os.path.expanduser('~'), 'jma_latest_run.json')

def _scan_latest(data_root, level):
    import glob, re
    parent = os.path.dirname(data_root) or '.'
    prefix = os.path.basename(data_root) + '_'
    pat = 'Lsurf_FD' if str(level) == 'surface' else f'Lp{level}_FD'
    cand = []
    for dd in glob.glob(os.path.join(parent, prefix + '*')):
        m = re.search(prefix + r'(\d{8})$', os.path.basename(dd))
        if not m:
            continue
        date = m.group(1)
        for hh in ('18','12','06','00'):
            lvl_dir = os.path.join(dd, hh, str(level))
            if os.path.isdir(lvl_dir) and glob.glob(os.path.join(lvl_dir, f'*{pat}*_grib2.bin')):
                cand.append((date, hh))
    if not cand:
        return None
    cand.sort()
    return data_root, cand[-1][0], cand[-1][1]

def resolve():
    if os.path.isfile(POINTER_PATH):
        try:
            with open(POINTER_PATH) as f:
                d = json.load(f)
            return d['data_root'], d['run_date'], d['run_hour'], 'pointer'
        except Exception as e:
            print('Pointer unreadable, scanning instead:', e)
    r = _scan_latest(DEFAULT_DATA_ROOT, LEVEL_HPA)
    if r:
        return r[0], r[1], r[2], 'scan'
    return None

_r = resolve()
if _r is None:
    DATA_DIR = None; OUTPUT_DIR = None
    print('WARNING: no pointer and no data found under', DEFAULT_DATA_ROOT + '_*')
    print('Either run the downloader, or set DEFAULT_DATA_ROOT to your data location.')
else:
    DATA_ROOT, RUN_DATE, RUN_HOUR, how = _r
    DATA_DIR   = os.path.join(f'{DATA_ROOT}_{RUN_DATE}', RUN_HOUR, str(LEVEL_HPA))
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), 'plots', f'{RUN_DATE}_{RUN_HOUR}', PRODUCT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'Run ({how}): {RUN_DATE} {RUN_HOUR}z')
    print(f'DATA_DIR   = {DATA_DIR}')
    print(f'OUTPUT_DIR = {OUTPUT_DIR}')
    if not os.path.isdir(DATA_DIR):
        print(f'WARNING: {DATA_DIR} not found — was level {LEVEL_HPA} downloaded for this run?')
##%%
# ---- borders ----
_countries = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
_provinces = shapereader.natural_earth(resolution='10m', category='cultural', name='admin_1_states_provinces')
_lakes = shapereader.natural_earth(resolution='10m', category='physical', name='lakes')

BORDER_SIMPLIFY = 0.03
PROV_SIMPLIFY   = 0.06
COAST_SIMPLIFY  = 0.03

def _simplify(geoms, tol):
    out = []
    for g in geoms:
        try:
            out.append(g.simplify(tol, preserve_topology=True))
        except Exception:
            out.append(g)
    return out

def _iran_country_geoms():
    geoms = []
    for rec in shapereader.Reader(_countries).records():
        name = (rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME') or '')
        adm = (rec.attributes.get('ADMIN') or '')
        if 'Iran' in name or 'Iran' in adm:
            geoms.append(rec.geometry)
    return geoms

def _iran_province_geoms():
    geoms = []
    for rec in shapereader.Reader(_provinces).records():
        at = rec.attributes
        if ('Iran' in str(at.get('admin','')) or 'Iran' in str(at.get('geonunit',''))
                or str(at.get('iso_a2',''))=='IR' or str(at.get('adm0_a3',''))=='IRN'
                or str(at.get('sr_adm0_a3',''))=='IRN'):
            geoms.append(rec.geometry)
    return geoms

def _caspian_geom():
    geoms = []
    for rec in shapereader.Reader(_lakes).records():
        nm = str(rec.attributes.get('name','') or rec.attributes.get('name_alt',''))
        if 'Caspian' in nm:
            geoms.append(rec.geometry)
    return geoms

IRAN_COUNTRY   = _simplify(_iran_country_geoms(), BORDER_SIMPLIFY)
IRAN_PROVINCES = _simplify(_iran_province_geoms(), PROV_SIMPLIFY)
CASPIAN        = _simplify(_caspian_geom(), COAST_SIMPLIFY)
print(f'Iran parts: {len(IRAN_COUNTRY)}   provinces: {len(IRAN_PROVINCES)}   Caspian polys: {len(CASPIAN)}')

def add_borders(ax):
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray')
    coast = cfeature.NaturalEarthFeature('physical', 'coastline', '10m')
    ax.add_feature(coast, linewidth=1.5, edgecolor='black', facecolor='none', zorder=4)
    if CASPIAN:
        ax.add_geometries(CASPIAN, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.5, zorder=4, joinstyle='round', capstyle='round')
    if IRAN_PROVINCES:
        ax.add_geometries(IRAN_PROVINCES, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                          linewidth=1.2, zorder=5, joinstyle='round', capstyle='round')
    ax.add_geometries(IRAN_COUNTRY, ccrs.PlateCarree(), facecolor='none', edgecolor='black',
                      linewidth=1.8, zorder=6, joinstyle='round', capstyle='round')
##%%
# ---- helpers ----
def fd_to_hours(fd):
    return int(fd[:2]) * 24 + int(fd[2:])

def list_level_files(data_dir):
    files = glob.glob(os.path.join(data_dir, f'*Lp{LEVEL_HPA}_FD*_grib2.bin'))
    out = []
    for f in files:
        try:
            fd = os.path.basename(f).split('FD')[1].split('_')[0]
            out.append((fd_to_hours(fd), f))
        except (IndexError, ValueError):
            pass
    out.sort()
    return out

def read_fields(path):
    # gh (gpm), t (degC), u, v (m/s); detected by paramId or CF name
    gh = t = u = v = None
    lats = lons = valid = run = None
    dss = cfgrib.open_datasets(path)
    for ds in dss:
        for vname, da in ds.data_vars.items():
            a = da.attrs
            pid = a.get('GRIB_paramId')
            sn = (a.get('GRIB_shortName') or '').lower()
            cf = (a.get('GRIB_cfName') or '').lower()
            if (pid == 131 or sn == 'u' or cf == 'eastward_wind') and u is None:
                u = np.asarray(da.values)
                lats = ds['latitude'].values
                lons = ds['longitude'].values
                valid = da['valid_time'].values if 'valid_time' in da.coords else ''
                run   = da['time'].values if 'time' in da.coords else ''
            elif (pid == 132 or sn == 'v' or cf == 'northward_wind') and v is None:
                v = np.asarray(da.values)
            elif (pid == 156 or sn == 'gh' or cf == 'geopotential_height') and gh is None:
                gh = np.asarray(da.values)
            elif (pid == 129 or sn == 'z' or cf == 'geopotential') and gh is None:
                gh = np.asarray(da.values) / 9.80665
            elif (pid == 130 or sn == 't' or cf == 'air_temperature') and t is None:
                t = np.asarray(da.values)
    if u is None or v is None:
        return None
    t_C = None
    if t is not None:
        t_C = t - 273.15 if np.nanmean(t) > 100 else t
    return gh, t_C, u, v, lats, lons, valid, run

def relative_vorticity(u, v, lons, lats):
    # zeta = dv/dx - du/dy (s^-1), metre spacing; dx shrinks with latitude
    R = 6371000.0
    latr = np.deg2rad(lats)
    dlon = np.deg2rad(np.gradient(lons))
    dlat = np.deg2rad(np.gradient(lats))
    dx = R * np.cos(latr)[:, None] * dlon[None, :]
    dy = (R * dlat)[:, None] * np.ones((1, len(lons)))
    dudy = np.gradient(u, axis=0) / dy
    dvdx = np.gradient(v, axis=1) / dx
    return dvdx - dudy
##%%
# ---- (run once) inspect variables in a 500 hPa file ----
_files = list_level_files(DATA_DIR)
print(f'{len(_files)} files at {LEVEL_HPA} hPa')
if _files:
    _h, _p = _files[0]
    print(f'\nVariables in {os.path.basename(_p)}:')
    for ds in cfgrib.open_datasets(_p):
        for vname, da in ds.data_vars.items():
            a = da.attrs
            print(f"  {vname:<8} paramId={a.get('GRIB_paramId')} shortName={a.get('GRIB_shortName')} cfName={a.get('GRIB_cfName')} units={a.get('GRIB_units')}")
##%%
# ---- vorticity colormap (x10^-5 s^-1) ----
vort_levels = [-20, -15, -10, -7, -5, -3, -1, 1, 3, 5, 7, 10, 15, 20]
vort_cmap = plt.get_cmap('RdBu_r')
vort_norm = mcolors.BoundaryNorm(vort_levels, vort_cmap.N)
##%%
# ---- plot ----
def fmt_utc(t):
    try:
        dt = np.datetime64(t, 'h')
        return f"{str(dt).replace('T', ' ')}:00 UTC"
    except Exception:
        return str(t)

if DATA_DIR is None:
    print('No data resolved — fix CONFIG and re-run.')
    files = []
else:
    files = list_level_files(DATA_DIR)
print(f'{len(files)} files found')

for hour, path in files:
    rec = read_fields(path)
    if rec is None:
        print(f'+{hour:03d}h : U/V not found, skipping')
        continue
    gh, t_C, u, v, lats, lons, valid, run = rec
    zeta = relative_vorticity(u, v, lons, lats) * 1e5

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(11, 12), dpi=200)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eef4fb')
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4)
    gl.top_labels = gl.right_labels = False

    # light smoothing for a clean look on the 0.5deg grid, then contourf.
    # transform_first=True avoids the cartopy/shapely 'GeometryCollection'
    # contourf bug while keeping smooth filled bands.
    zeta_s = gaussian_filter(np.nan_to_num(zeta), sigma=1.0)
    lon2d, lat2d = np.meshgrid(lons, lats)   # 2-D coords for transform_first
    cf = ax.contourf(lon2d, lat2d, zeta_s, levels=vort_levels, cmap=vort_cmap,
                     norm=vort_norm, extend='both', transform=ccrs.PlateCarree(),
                     transform_first=True)
    cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, aspect=50, ticks=vort_levels)
    cbar.set_label(CREDIT)

    if gh is not None:
        lo = np.floor(np.nanmin(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hi = np.ceil(np.nanmax(gh)/HEIGHT_STEP)*HEIGHT_STEP
        hlev = np.arange(lo, hi+HEIGHT_STEP, HEIGHT_STEP)
        # Convert to DECAMETERS first, then contour, so labels are plain integers
        # using the same '%d' mechanism that already works for the temperature lines.
        gh_dkm = gh / 10.0
        hlev_dkm = hlev / 10.0
        csh = ax.contour(lon2d, lat2d, gh_dkm, levels=hlev_dkm, colors='black', linewidths=1.2,
                         transform=ccrs.PlateCarree(), transform_first=True)
        # Label every height contour. Try auto inline placement; if a contour gets no
        # label, force one manually near the map-centre longitude so NO line is unlabeled.
        lbls = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, inline_spacing=2)
        for _t in lbls:
            _t.set_zorder(12)
        # belt-and-suspenders: manual labels at the centre meridian for any line that
        # the auto-placer skipped (long near-horizontal contours sometimes get skipped).
        try:
            cx = 0.5 * (lon_min + lon_max)
            ci = int(round(np.interp(cx, lons if np.ndim(lons)==1 else lons[0], np.arange(len(lons if np.ndim(lons)==1 else lons[0])))))
            ghcol = (gh[:, ci] / 10.0)
            latcol = lats if np.ndim(lats)==1 else lats[:, 0]
            man_pts = []
            for lv in hlev_dkm:
                d = ghcol - lv
                sgn = np.sign(d)
                xs = np.where(np.diff(sgn) != 0)[0]
                if len(xs):
                    k = xs[len(xs)//2]
                    if lat_min <= latcol[k] <= lat_max:
                        man_pts.append((cx, float(latcol[k])))
            if man_pts:
                ml = ax.clabel(csh, fmt='%d', fontsize=8, inline=True, manual=man_pts)
                for _t in ml:
                    _t.set_zorder(12)
        except Exception as _e:
            print('manual height-label fallback skipped:', _e)

    if t_C is not None:
        lo = np.floor(np.nanmin(t_C)/TEMP_STEP)*TEMP_STEP
        hi = np.ceil(np.nanmax(t_C)/TEMP_STEP)*TEMP_STEP
        tlev = np.arange(lo, hi+TEMP_STEP, TEMP_STEP)
        cst = ax.contour(lons, lats, t_C, levels=tlev, colors='green', linewidths=1.0,
                         linestyles='dashed', transform=ccrs.PlateCarree())
        ax.clabel(cst, fmt='%d', fontsize=6, inline=True)

    add_borders(ax)

    ax.set_title(
        f'JMA GSM {PARAM_NAME} ({PARAM_UNIT}) (Shaded)\n'
        f'+ Height contours (dkm) + Temperature (°C, green)\n'
        f'Run Time: {fmt_utc(run)}    Valid Time: {fmt_utc(valid)}    Forecast Hour: +{hour:03d}h',
        fontsize=11)

    out = os.path.join(OUTPUT_DIR, f'vort500_FD{hour:03d}h.png')
    plt.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f'+{hour:03d}h : saved {out}  (zeta {np.nanmin(zeta):.1f}..{np.nanmax(zeta):.1f})')
##%%
# ---- build GIF ----
import imageio.v2 as imageio
imgs = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png'))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in imgs]
gif_path = os.path.join(OUTPUT_DIR, 'vorticity_500.gif')
imageio.mimsave(gif_path, frames, duration=0.4)
print('Created GIF:', gif_path)